# AI Social Media Agent — Prompt Template Pipeline
## Abhinav Gupta | IITB Capstone 2026

---

## SECTION 1: SETUP & CONFIGURATION

In [ ]:
# Cell 1 — Install required libraries
# Run once, then comment out
# !pip install langchain langchain-anthropic langchain-openai openai anthropic -q
# !pip install python-dotenv transformers torch faiss-cpu pdfplumber chromadb -q
print("✅ Libraries ready!")

In [ ]:
# Cell 2 — All imports in one place
import os
import re
import json
import random
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import HumanMessage
from transformers import pipeline as hf_pipeline

print("✅ All imports loaded!")

In [ ]:
# Cell 3 — Load API keys securely from .env file
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
env_path = os.path.join(repo_root, '.env')
load_dotenv(dotenv_path=env_path)

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if ANTHROPIC_API_KEY:
    print("Anthropic key loaded:", ANTHROPIC_API_KEY[:10] + "...")
else:
    print("❌ Anthropic key NOT found — check your .env file")

if OPENAI_API_KEY:
    print("OpenAI key loaded:", OPENAI_API_KEY[:10] + "...")
else:
    print("❌ OpenAI key NOT found — check your .env file")

In [ ]:
# Cell 4 — Initialize LLM (Claude Haiku for dev, Sonnet for production)
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    api_key=ANTHROPIC_API_KEY,
    max_tokens=4000
)

# Quick test
response = llm.invoke([HumanMessage(content="Say hello in 5 words.")])
print("LLM test:", response.content)
print("✅ LLM initialized!")

In [ ]:
# Cell 5 — Load DistilBERT sentiment model (loads once per session)
print("Loading DistilBERT sentiment model (~15 seconds)...")
sentiment_analyzer = hf_pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)
print("✅ DistilBERT loaded!")

---

## SECTION 2: PROMPT TEMPLATES

Three generations of templates to demonstrate optimization progression:
- **Level 1:** Zero-shot (no examples, no reasoning)
- **Level 2:** Few-shot (with real brand examples)
- **Level 3:** Unified (few-shot + chain-of-thought + JSON output)

Use these to show examiners how each optimization improves quality.

In [ ]:
# Cell 6 — LEVEL 1: Zero-Shot Templates (baseline — no examples, no COT)
# Used for comparison to show improvement from few-shot and COT

zeroshot_template = PromptTemplate(
    input_variables=["brand_name", "topic", "tone", "platform"],
    template="""
You are an expert social media content creator.

Generate exactly 5 distinct social media post variants for the brand {brand_name}.

PLATFORM: {platform}
TOPIC: {topic}
TONE: {tone}

PLATFORM RULES:
- Twitter: maximum 280 characters, 2-3 hashtags, punchy and direct
- LinkedIn: 150-300 words, professional but human, 3-5 hashtags
- Instagram: storytelling style, 5-10 hashtags, strong opening hook

OUTPUT FORMAT:
Return exactly 5 posts numbered like this:
1. [post text] [hashtags]
2. [post text] [hashtags]
3. [post text] [hashtags]
4. [post text] [hashtags]
5. [post text] [hashtags]

Write only the posts. No explanations.
"""
)

print("✅ Zero-shot template loaded!")

In [ ]:
# Cell 7 — Few-shot examples library
# Sources: Nike (Twitter), Adobe (LinkedIn), NatGeo (Instagram)
# Real posts from real brands — not fabricated

twitter_examples = """
EXAMPLE 1 (Athlete/Event Celebration - Nike):
"When you're this fast, you don't ask for permission. Jutta Leerdam
breaks the Olympic record in the Speed Skating 1000m and wins her
first Gold. #MilanoCortina2026 #Olympics"

EXAMPLE 2 (Product Launch - Nike):
"Mute the gallery. Introducing Nike Powerbeats Pro 2, the ultimate
fitness earbuds. The first-ever Beats x Nike collab — where premium
sound meets unbeatable stability. No advice required.
Launching March 20th at 9am PST."

EXAMPLE 3 (Motivational/Quote - Nike):
"There's no failure in sports. It's steps to success." - @Giannis_An34
Regardless of the outcome, there's always a reward ahead. #AlwaysForward
"""

linkedin_examples = """
EXAMPLE 1 (Thought Leadership - Adobe):
"Creative teams are under pressure to deliver more content across more
channels than ever before. Many are turning to AI to handle production
tasks like resizing, versioning, and early-stage ideation, so they can
spend more time refining concepts and craft. Adobe's latest research
breaks down how this shift is changing the way we do creative work."

EXAMPLE 2 (Corporate Announcement - Adobe):
"The way brands are discovered is changing fast. Today, Adobe completed
its acquisition of Semrush, expanding how businesses show up, get found,
and drive growth in an AI-first world. AI-driven traffic to U.S. retail
sites is up 269% year over year, yet most businesses still have significant
gaps in how they appear across AI surfaces."

EXAMPLE 3 (Community/Human Story - Adobe):
"My greatest inspiration came from being raised by a self-made single
mom who showed us strength and resilience. From drawing at a young age
to now utilizing Adobe tools to bridge the gap between art and business,
Alexandra Yvette continues to create and share her talents with the
world this Women's History Month."
"""

instagram_examples = """
EXAMPLE 1 (Wildlife/Action Story - NatGeo):
"This image was taken near Polán, Toledo province (Spain) last August,
from a tiny hide-out overlooking a small waterhole where lynxes
occasionally come down to drink. It was an extremely hot afternoon,
and a rabbit was very close to the water. Suddenly, a lynx appeared
silently, but the rabbit noticed it at the very last second.
Photo by @alexandrovich_yo from our @natgeoyourshot community."

EXAMPLE 2 (Landscape/Philosophical - NatGeo):
"From Punta Helbronner, 11,370 feet above sea level on the Italian
side of Mont Blanc in the Alps, my guide walks out across the snow
toward Dent du Géant, the Giant's Tooth, illuminated only by the
light of my drone during a long exposure. Photos by @reuben"

EXAMPLE 3 (Milestone/Achievement - NatGeo):
"Barcelona's Sagrada Família has reached a new milestone. With the
cross now installed atop its central Jesus tower, the basilica stands
more than 560 feet (170m) — surpassing Germany's Ulm Minster as the
world's tallest church. Photographs by Nuria Puentes, National Geographic-España"
"""

def get_examples(platform):
    if platform == "twitter":
        return twitter_examples
    elif platform == "linkedin":
        return linkedin_examples
    elif platform == "instagram":
        return instagram_examples
    else:
        return ""

print("✅ Few-shot examples library loaded!")

In [ ]:
# Cell 8 — LEVEL 2: Few-Shot Template (with real brand examples)
# Improvement over zero-shot: Claude has concrete targets to aim for

fewshot_template = PromptTemplate(
    input_variables=["brand_name", "topic", "tone", "platform", "examples"],
    template="""
You are an expert social media content creator.

Generate exactly 5 distinct social media post variants for the brand {brand_name}.

PLATFORM: {platform}
TOPIC: {topic}
TONE: {tone}

PLATFORM RULES:
- Twitter: maximum 280 characters, 2-3 hashtags, punchy and direct
- LinkedIn: 150-300 words, professional but human, 3-5 hashtags
- Instagram: storytelling style, 5-10 hashtags, strong opening hook

HIGH PERFORMING EXAMPLES FOR REFERENCE:
{examples}

Now write 5 posts matching the same quality and style as the examples above.

OUTPUT FORMAT:
Return exactly 5 posts numbered like this:
1. [post text] [hashtags]
2. [post text] [hashtags]
3. [post text] [hashtags]
4. [post text] [hashtags]
5. [post text] [hashtags]

Write only the posts. No explanations.
"""
)

print("✅ Few-shot template loaded!")

In [ ]:
# Cell 9 — LEVEL 3: Unified Template (few-shot + COT + JSON output)
# Our production template — best quality output

unified_template = PromptTemplate(
    input_variables=["brand_name", "topic", "tone", "platform", "examples"],
    template="""
You are an expert social media content creator specializing in brand voice alignment.

Generate exactly 5 distinct social media post variants for the brand {brand_name}.

PLATFORM: {platform}
TOPIC: {topic}
TONE: {tone}

PLATFORM RULES:
- Twitter: maximum 280 characters, 2-3 hashtags, punchy and direct
- LinkedIn: 150-300 words, professional but human, 3-5 hashtags, short paragraphs
- Instagram: storytelling style, strong opening hook, 5-10 hashtags, emojis throughout

HIGH PERFORMING EXAMPLES FOR REFERENCE:
{examples}

THINK STEP BY STEP BEFORE WRITING EACH VARIANT:
Step 1 - Audience: Who is the primary audience for this variant?
Step 2 - Key Message: What is the single most important message?
Step 3 - Tone Check: Does the tone match the brand and platform?
Step 4 - Hook: What opening line will stop the scroll?
Step 5 - CTA: What action should the reader take?

Make each variant structurally distinct:
- Variant 1: Lead with business impact
- Variant 2: Lead with customer benefit
- Variant 3: Lead with a bold statement or question
- Variant 4: Lead with data or proof points
- Variant 5: Lead with a human or emotional angle

IMPORTANT: Respond with ONLY a valid JSON array.
No explanations. No preamble. No markdown. Just the JSON.

Return exactly this structure:
[
  {{
    "variant_id": 1,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 2,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 3,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 4,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }},
  {{
    "variant_id": 5,
    "reasoning": "brief explanation of thinking for this variant",
    "post_text": "the post content here",
    "hashtags": ["#hashtag1", "#hashtag2"],
    "tone": "describe the tone used",
    "suggested_posting_time": "best day and time to post",
    "platform": "{platform}"
  }}
]
"""
)

print("✅ Unified production template loaded!")

---

## SECTION 3: PIPELINE FUNCTIONS

Core functions for generating and parsing posts.

In [ ]:
# Cell 10 — JSON parser helper
def parse_json_response(response_text):
    """
    Cleans and parses Claude's JSON response
    Handles markdown code blocks if present
    """
    text = response_text.strip()
    if text.startswith("```"):
        text = text.split("```")[1]
        if text.startswith("json"):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing failed: {e}")
        return None

print("✅ JSON parser ready!")

In [ ]:
# Cell 11 — Main pipeline function
def generate_posts(brand_name, topic, tone, platform):
    """
    Main pipeline — generates 5 post variants using
    unified template (few-shot + COT + JSON)

    Args:
        brand_name (str): e.g. "Adobe"
        topic (str): e.g. "Adobe's acquisition of Semrush"
        tone (str): e.g. "professional, forward looking"
        platform (str): "twitter", "linkedin", "instagram"

    Returns:
        list: 5 structured post variants as Python list of dicts
    """
    examples = get_examples(platform)

    filled_prompt = unified_template.format(
        brand_name=brand_name,
        topic=topic,
        tone=tone,
        platform=platform,
        examples=examples
    )

    response = llm.invoke([HumanMessage(content=filled_prompt)])
    return parse_json_response(response.content)

print("✅ generate_posts() function ready!")

---

## SECTION 4: FEATURE EXTRACTION & SCORING

Extracting engagement features from posts and scoring them.
Mock scorer simulates Khushee's XGBoost model.
Swap `mock_predict_engagement` with `predict_engagement` 
from `modules/predictor.py` when Khushee's model is ready.

In [ ]:
# Cell 12 — Sentiment scorer using DistilBERT
def get_sentiment_score(text):
    """
    Returns sentiment score from -1 to 1
    Positive posts score closer to +1
    Negative posts score closer to -1
    Uses DistilBERT for contextual understanding
    """
    result = sentiment_analyzer(text[:512])[0]
    score = result['score']
    if result['label'] == 'POSITIVE':
        return round(score, 4)
    else:
        return round(-score, 4)

# Test it
test_score = get_sentiment_score("Adobe's acquisition of Semrush strengthens our market position!")
print(f"Test sentiment score: {test_score}")
print("✅ Sentiment scorer ready!")

In [ ]:
# Cell 13 — Feature extractor
def extract_features(variant, platform):
    """
    Extracts content quality features from a post variant.
    Posting time is NOT included — it is a recommendation,
    not a quality signal. Scoring posting time would
    penalize good posts for having off-peak suggested times.

    Args:
        variant: single post dict from generate_posts()
        platform: "twitter", "linkedin", "instagram"

    Returns:
        dict of 8 content quality features
    """
    post_text = variant['post_text']
    hashtags = variant['hashtags']

    # Feature 1 — Caption length (characters)
    caption_length = len(post_text)

    # Feature 2 — Hashtag count
    hashtag_count = len(hashtags)

    # Feature 3 — Sentiment score (DistilBERT, -1 to 1)
    sentiment_score = get_sentiment_score(post_text)

    # Feature 4 — Has CTA (call to action)
    cta_keywords = [
        'click', 'learn more', 'discover', 'explore',
        'try', 'get', 'join', 'sign up', 'download',
        'shop', 'buy', 'visit', 'check out', 'read',
        'watch', 'follow', 'share', 'comment', 'tag'
    ]
    has_cta = int(any(
        keyword in post_text.lower()
        for keyword in cta_keywords
    ))

    # Feature 5 — Platform encoded as integer
    platform_map = {"twitter": 0, "linkedin": 1, "instagram": 2}
    platform_encoded = platform_map.get(platform, 0)

    # Feature 6 — Word count
    word_count = len(post_text.split())

    # Feature 7 — Has question (drives comments/engagement)
    has_question = int('?' in post_text)

    # Feature 8 — Has emoji
    emoji_pattern = re.compile(
        "[" u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F9FF"
        u"\U00002700-\U000027BF" "]+",
        flags=re.UNICODE
    )
    has_emoji = int(bool(emoji_pattern.search(post_text)))

    return {
        "caption_length": caption_length,
        "hashtag_count": hashtag_count,
        "sentiment_score": sentiment_score,
        "has_cta": has_cta,
        "platform": platform_encoded,
        "word_count": word_count,
        "has_question": has_question,
        "has_emoji": has_emoji
    }

print("✅ Feature extractor ready!")

In [ ]:
# Cell 14 — Mock engagement scorer
# TEMPORARY: Simulates Khushee's XGBoost predict_engagement()
# TO REPLACE: from modules.predictor import predict_engagement

def mock_predict_engagement(features_dict):
    """
    Mock of Khushee's XGBoost engagement predictor.
    Uses feature heuristics to simulate scoring.

    Replace this with:
        from modules.predictor import predict_engagement
    when Khushee's model is ready in Week 3.
    """
    score = 0.5  # base score

    # Sentiment is a strong engagement driver
    score += features_dict['sentiment_score'] * 0.15

    # CTA boosts engagement
    if features_dict['has_cta']:
        score += 0.08

    # Questions drive comments
    if features_dict['has_question']:
        score += 0.06

    # Emojis boost Instagram and Twitter
    if features_dict['has_emoji']:
        if features_dict['platform'] in [0, 2]:  # twitter, instagram
            score += 0.05

    # Optimal hashtag count per platform
    hashtags = features_dict['hashtag_count']
    platform = features_dict['platform']

    if platform == 0:      # Twitter: 2-3 optimal
        if 2 <= hashtags <= 3:
            score += 0.05
        elif hashtags > 5:
            score -= 0.05
    elif platform == 1:    # LinkedIn: 3-5 optimal
        if 3 <= hashtags <= 5:
            score += 0.05
        elif hashtags > 7:
            score -= 0.05
    elif platform == 2:    # Instagram: 5-10 optimal
        if 5 <= hashtags <= 10:
            score += 0.05
        elif hashtags > 15:
            score -= 0.05

    # NOTE: posting_hour deliberately excluded
    # Posting time is a recommendation output,
    # not a content quality signal

    # Small random noise — real engagement is not deterministic
    score += random.uniform(-0.03, 0.03)

    # Clamp between 0 and 1
    return round(max(0.0, min(1.0, score)), 4)

print("✅ Mock scorer ready!")

In [ ]:
# Cell 15 — Optimizer function
def optimize_variants(variants, platform):
    """
    Scores and ranks 5 post variants by predicted engagement.
    Flags the top variant as recommended.

    Args:
        variants: list of 5 post dicts from generate_posts()
        platform: "twitter", "linkedin", "instagram"

    Returns:
        ranked_variants: list sorted best first
        scores: list of raw scores
    """
    print("Extracting features and scoring variants...")
    scores = []

    for i, variant in enumerate(variants):
        features = extract_features(variant, platform)
        score = mock_predict_engagement(features)
        scores.append(score)
        variant['predicted_score'] = score
        variant['features'] = features
        variant['is_recommended'] = False
        print(f"  Variant {i+1}: score = {score}")

    # Sort descending by score
    ranked = sorted(
        variants,
        key=lambda x: x['predicted_score'],
        reverse=True
    )

    # Flag top variant as recommended
    ranked[0]['is_recommended'] = True

    return ranked, scores

print("✅ Optimizer ready!")

---

## SECTION 5: COMPARISON DEMO

**For examiners — shows how each optimization improves quality.**

Run these 3 cells with the same topic and platform to compare:
- Cell 16: Zero-shot output (baseline)
- Cell 17: Few-shot output (with examples)
- Cell 18: Unified output (few-shot + COT + JSON + scoring)

In [ ]:
# Cell 16 — COMPARISON: Zero-Shot Output (baseline)
# Same topic and platform for fair comparison

DEMO_BRAND = "Adobe"
DEMO_TOPIC = "Adobe's acquisition of Semrush"
DEMO_TONE = "professional, strategic, forward looking"
DEMO_PLATFORM = "linkedin"

print("=" * 60)
print("LEVEL 1: ZERO-SHOT (no examples, no reasoning)")
print("=" * 60)

zeroshot_filled = zeroshot_template.format(
    brand_name=DEMO_BRAND,
    topic=DEMO_TOPIC,
    tone=DEMO_TONE,
    platform=DEMO_PLATFORM
)

zeroshot_response = llm.invoke([HumanMessage(content=zeroshot_filled)])
print(zeroshot_response.content)

In [ ]:
# Cell 17 — COMPARISON: Few-Shot Output
# Same topic — now with real brand examples

print("=" * 60)
print("LEVEL 2: FEW-SHOT (with Nike/Adobe/NatGeo examples)")
print("=" * 60)

fewshot_filled = fewshot_template.format(
    brand_name=DEMO_BRAND,
    topic=DEMO_TOPIC,
    tone=DEMO_TONE,
    platform=DEMO_PLATFORM,
    examples=get_examples(DEMO_PLATFORM)
)

fewshot_response = llm.invoke([HumanMessage(content=fewshot_filled)])
print(fewshot_response.content)

In [ ]:
# Cell 18 — COMPARISON: Unified Output (few-shot + COT + JSON + scoring)
# Same topic — now with full pipeline including optimization

print("=" * 60)
print("LEVEL 3: UNIFIED (few-shot + COT + JSON + optimizer)")
print("=" * 60)

# Generate 5 variants using full pipeline
variants = generate_posts(
    brand_name=DEMO_BRAND,
    topic=DEMO_TOPIC,
    tone=DEMO_TONE,
    platform=DEMO_PLATFORM
)

if variants:
    # Score and rank variants
    ranked_variants, scores = optimize_variants(variants, DEMO_PLATFORM)

    print("\n" + "=" * 60)
    print("OPTIMIZATION RESULTS — RANKED BY PREDICTED ENGAGEMENT")
    print("=" * 60)

    for i, variant in enumerate(ranked_variants):
        recommended = "⭐ RECOMMENDED" if variant['is_recommended'] else ""
        print(f"\nRank {i+1} — Score: {variant['predicted_score']} {recommended}")
        print(f"Reasoning: {variant.get('reasoning', 'N/A')[:80]}...")
        print(f"Preview: {variant['post_text'][:100]}...")
        print(f"Hashtags: {' '.join(variant['hashtags'])}")
        print(f"Suggested posting time: {variant['suggested_posting_time']}")

    print("\n" + "=" * 60)
    print("RECOMMENDED POST — FULL TEXT:")
    print("=" * 60)
    print(ranked_variants[0]['post_text'])
    print(f"\nHashtags: {' '.join(ranked_variants[0]['hashtags'])}")
    print(f"Predicted engagement score: {ranked_variants[0]['predicted_score']}")
    print(f"Suggested posting time: {ranked_variants[0]['suggested_posting_time']}")
else:
    print("❌ Generation failed")

---

## SECTION 6: FULL PIPELINE TEST

Test the complete pipeline across all 3 platforms and multiple brands.

In [ ]:
# Cell 19 — Test across all 3 platforms
test_brand = "Adobe"
test_topic = "Adobe's acquisition of Semrush"
test_tone = "professional, strategic, forward looking"

print("FULL PIPELINE TEST — ALL 3 PLATFORMS")
print("=" * 60)

all_results = {}

for platform in ["twitter", "linkedin", "instagram"]:
    print(f"\nGenerating for {platform.upper()}...")

    variants = generate_posts(test_brand, test_topic, test_tone, platform)

    if variants:
        ranked, scores = optimize_variants(variants, platform)
        all_results[platform] = ranked

        print(f"✅ {platform.upper()} — {len(variants)} variants generated")
        print(f"   Best score: {ranked[0]['predicted_score']}")
        print(f"   Recommended: {ranked[0]['post_text'][:80]}...")
    else:
        print(f"❌ {platform.upper()} — generation failed")

print("\n" + "=" * 60)
print(f"SUMMARY: {len(all_results)}/3 platforms successful")

In [ ]:
# Cell 20 — Test with different brand (try Adidas or Duolingo)
# Change brand_name and tone to see different outputs

variants_adidas = generate_posts(
    brand_name="Adidas",
    topic="launching new running shoe",
    tone="bold, performance-driven, athlete-focused",
    platform="instagram"
)

if variants_adidas:
    ranked_adidas, _ = optimize_variants(variants_adidas, "instagram")
    print("ADIDAS — INSTAGRAM — RECOMMENDED POST:")
    print("=" * 60)
    print(ranked_adidas[0]['post_text'])
    print(f"\nHashtags: {' '.join(ranked_adidas[0]['hashtags'])}")
    print(f"Score: {ranked_adidas[0]['predicted_score']}")

---

## SECTION 7: INTEGRATION NOTES FOR TEAM

### Replacing Mock Scorer with Khushee's Model
When Khushee delivers `modules/predictor.py`:

```python
# Replace in Cell 14:
# from modules.predictor import predict_engagement
# Use predict_engagement() instead of mock_predict_engagement()
```

### Integrating Shreyas's RAG Module
When Shreyas delivers `modules/rag.py`:

```python
# Add brand_context parameter to generate_posts()
# brand_context = shreyas_rag.retrieve(topic, brand_name)
# Add {brand_context} field to unified_template
```

### Integrating Shreyas's Trending Topics
When Shreyas delivers `modules/trending.py`:

```python
# Add trending_topics parameter to generate_posts()
# trending = shreyas_trends.get_trending(topic)
# Add {trending_topics} field to unified_template
```

### JSON Schema for Aditya's Streamlit UI
Each variant in the output contains:
```json
{
  "variant_id": 1,
  "reasoning": "Claude's thinking for this variant",
  "post_text": "the actual post content",
  "hashtags": ["#hashtag1", "#hashtag2"],
  "tone": "tone used",
  "suggested_posting_time": "Tuesday 9:00 AM ET",
  "platform": "linkedin",
  "predicted_score": 0.84,
  "is_recommended": true
}
```